# 01 — Exploratory Data Analysis
Understand the raw Online Retail II dataset before any cleaning.

## 1. Imports & config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


## 2. Load raw data

In [ ]:
RAW = Path("../data/raw/online_retail_II.xlsx")

df = pd.read_excel(RAW, sheet_name=None)   # loads both sheets
print("Sheets found:", list(df.keys()))

df = pd.concat(df.values(), ignore_index=True)
print(f"Combined shape: {df.shape}")
df.head()


## 3. Basic info

In [ ]:
df.info()


In [ ]:
df.describe(include="all")


## 4. Missing values

In [ ]:
missing = df.isnull().sum().rename("Missing").to_frame()
missing["Pct"] = (missing["Missing"] / len(df) * 100).round(2)
missing[missing["Missing"] > 0].sort_values("Pct", ascending=False)


## 5. Duplicate rows

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes:,}  ({dupes/len(df)*100:.2f}%)")


## 6. Cancellations

In [ ]:
df["IsCancelled"] = df["Invoice"].astype(str).str.startswith("C")
print(df["IsCancelled"].value_counts())
print(f"Cancellation rate: {df['IsCancelled'].mean()*100:.2f}%")


## 7. Date range

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print("Earliest:", df["InvoiceDate"].min())
print("Latest  :", df["InvoiceDate"].max())
print("Span    :", (df["InvoiceDate"].max() - df["InvoiceDate"].min()).days, "days")


## 8. Revenue distribution

In [ ]:
df_valid = df[~df["IsCancelled"] & df["CustomerID"].notna()].copy()
df_valid["Revenue"] = df_valid["Quantity"] * df_valid["Price"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_valid["Revenue"].clip(0, 500).hist(bins=60, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Revenue per line item (clipped at £500)")
axes[0].set_xlabel("Revenue (£)")

df_valid.groupby("Invoice")["Revenue"].sum().clip(0, 2000).hist(bins=60, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Order value distribution (clipped at £2000)")
axes[1].set_xlabel("Order Total (£)")
plt.tight_layout()
plt.show()


## 9. Top 10 countries by revenue

In [ ]:
top_countries = (
    df_valid.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_countries.plot(kind="barh", figsize=(10, 5), color="steelblue", edgecolor="white")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Total Revenue (£)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 10. Monthly revenue trend

In [ ]:
df_valid["YearMonth"] = df_valid["InvoiceDate"].dt.to_period("M")
monthly = df_valid.groupby("YearMonth")["Revenue"].sum()

monthly.plot(figsize=(12, 4), marker="o", color="steelblue")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 11. Orders by day of week

In [ ]:
df_valid["DayOfWeek"] = df_valid["InvoiceDate"].dt.day_name()
order_day = df_valid.groupby("DayOfWeek")["Invoice"].nunique()
order_day = order_day.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])

order_day.plot(kind="bar", figsize=(10, 4), color="mediumpurple", edgecolor="white")
plt.title("Unique Orders by Day of Week")
plt.ylabel("Order Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 12. Customer coverage

In [ ]:
total_customers = df["CustomerID"].nunique()
customers_with_id = df_valid["CustomerID"].nunique()
print(f"Total unique CustomerIDs (incl nulls): {total_customers:,}")
print(f"Customers with valid ID             : {customers_with_id:,}")
print(f"Coverage                            : {customers_with_id/total_customers*100:.1f}%")


## 13. Key observations
- Note missing CustomerID %, cancellation %, date range, skew in revenue
- These observations drive cleaning decisions in `etl/transform.py`